<a href="https://colab.research.google.com/github/biglalo104/Projects/blob/main/ML%20Breast%20Cancer%20Wisconsin%20Diagnosis%20using%20Logistic%20Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
#Loading libararies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [26]:
#Loading the dataset
data= pd.read_csv("data(1).csv")
print(data.head)

<bound method NDFrame.head of            id diagnosis  ...  fractal_dimension_worst  Unnamed: 32
0      842302         M  ...                  0.11890          NaN
1      842517         M  ...                  0.08902          NaN
2    84300903         M  ...                  0.08758          NaN
3    84348301         M  ...                  0.17300          NaN
4    84358402         M  ...                  0.07678          NaN
..        ...       ...  ...                      ...          ...
564    926424         M  ...                  0.07115          NaN
565    926682         M  ...                  0.06637          NaN
566    926954         M  ...                  0.07820          NaN
567    927241         M  ...                  0.12400          NaN
568     92751         B  ...                  0.07039          NaN

[569 rows x 33 columns]>


In [27]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    object 
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             5

In [28]:
data.drop(['Unnamed: 32','id'],axis=1,inplace=True)
data['diagnosis'] = data['diagnosis'].map({'M':1,'B':0})

In [29]:
y = data['diagnosis'].values
x_data = data.drop(['diagnosis'],axis=1)

In [30]:
#Normalization
x = (x_data - x_data.min())/(x_data.max()-x_data.min())

In [31]:
#Splitting data for training and testing
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size = 0.15,random_state = 42)

In [32]:
x_train =x_train.T
x_test = x_test.T
y_train = y_train.T
y_test = y_test.T

print("x train:",x_train.shape)
print("x test:",x_test.shape)
print("y train:",y_train.shape)
print("y test:",y_test.shape)

x train: (30, 483)
x test: (30, 86)
y train: (483,)
y test: (86,)


In [33]:
#Initialize Model Architecture
def initialize_weights_and_bias(dimension):
  w = np.random.randn(dimension,1) * 0.01
  b = 0.0
  return w,b

In [34]:
#Sigmoid Function to compute Z
def sigmoid(z):
  return 1/(1+np.exp(-z))

In [35]:
#Forward Backward Propagation
def forward_backward_propagation(w,b,x_train,y_train):
  m = x_train.shape[1]
  z = np.dot(w.T,x_train) + b
  y_head = sigmoid(z)

  cost =(-1/m)* np.sum(y_train * np.log(y_head) + (1-y_train) * np.log(1- y_head))

  derivative_weight = (1/m) * np.dot(x_train,(y_head - y_train).T)
  derivative_bias = (1/m) * np.sum(y_head - y_train)
  gradients = {"derivative_weight": derivative_weight,"derivative_bias": derivative_bias}
  return cost,gradients

In [36]:
#Updating Parametres
def update(w,b,x_train,y_train,learning_rate,num_iterations):
  costs =[]
  gradiemts ={}
  for i in range(num_iterations):
    cost,grad = forward_backward_propagation(w,b,x_train,y_train)
    w -= learning_rate * grad["derivative_weight"]
    b -= learning_rate * grad["derivative_bias"]

    if i % 100 == 0:
      costs.append(cost)
      print(f"Cost after iteration {i} is {cost}")


  parameters = {"weight":w, "bias": b}
  return parameters,gradients,costs

In [37]:
#Making Predictions
def predict(w,b,x_test):
  m = x_test.shape[1]
  y_prediction = np.zeros((1,m))
  z = sigmoid(np.dot(w.T,x_test) +b)


  for i in range(z.shape[1]):
    y_prediction[0,i] = 1 if z[0,i]> 0.5 else 0

    return y_prediction


In [38]:
#Implement Logistic Regression
def logistic_regression(x_train,y_train,x_test,y_test,learning_rate=0.01,num_iterations=1000):
  dimension = x_train.shape[0]
  w,b = initialize_weights_and_bias(dimension)
  parameters,gradients,costs = update(w,b,x_train,y_train,learning_rate,num_iterations)

  y_prediction_test = predict(parameters["weight"],parameters["bias"],x_test)
  y_prediction_train = predict(parameters["weight"],parameters["bias"],x_train)

  print(f"Train accuracy: {100 - np.mean(np.abs(y_prediction_train - y_train))*100}%")
  print(f"Test accuracy: {100 - np.mean(np.abs(y_prediction_test -y_test))*100}%")
